# 3. RecA Feature Selection and RFE Benchmark Workflow

Publication-ready notebook adapted from the RecA workflow and the senior PaDEL feature-selection notebook.

This notebook performs low-variance filtering, ANOVA F-score ranking, and optional Recursive Feature Elimination with cross-validation (RFECV) using SVM, Random Forest, and XGBoost estimators for *Mycobacterium tuberculosis* RecA inhibitor classification.

## Workflow overview

This notebook is designed to be run after the PaDEL fingerprint/modeling-matrix notebook.

**Main input**

`outputs/02_recA_modeling_matrix.csv`

**Main outputs**

- `outputs/feature_selection/03_recA_low_variance_filtered.csv`
- `outputs/feature_selection/03_recA_fscore_ranking.csv`
- `outputs/feature_selection/03_recA_top_fscore_dataset.csv`
- `outputs/feature_selection/rfe/03_recA_rfecv_selected_features_*.csv`
- `outputs/feature_selection/rfe/03_recA_rfecv_summary.csv`

The workflow follows the logic of the senior notebook: low-variance filtering → F-score selection → RFE/RFECV. The code is reorganized for RecA, reproducibility, and publication-quality documentation.

In [ ]:
# ============================================================
# Environment and library setup
# ============================================================

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif, RFECV
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
except Exception:
    XGBOOST_AVAILABLE = False

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("XGBoost available:", XGBOOST_AVAILABLE)


## 1. Configuration

Adjust only this section if your input file or feature-selection settings are different.

For publication and reproducibility, the default configuration avoids very aggressive RFE on thousands of descriptors. First, it reduces the matrix using variance filtering and F-score ranking, then RFECV is applied only to the preselected top descriptors.

In [ ]:
# ============================================================
# User configuration
# ============================================================

BASE_DIR = Path.cwd()

INPUT_FILE = BASE_DIR / "outputs" / "02_recA_modeling_matrix.csv"

OUTPUT_DIR = BASE_DIR / "outputs" / "feature_selection"
RFE_OUTPUT_DIR = OUTPUT_DIR / "rfe"
FIGURE_DIR = OUTPUT_DIR / "figures"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RFE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

# Metadata columns expected from the RecA PaDEL/modeling-matrix workflow.
# The code will keep only columns that exist in the uploaded dataset.
POSSIBLE_META_COLUMNS = [
    "Name",
    "molecule_chembl_id",
    "chembl_id",
    "canonical_smiles",
    "standard_type",
    "standard_value",
    "standard_units",
    "pEC50",
    "EC50_nM",
    "bioactivity_class",
    "class",
]

TARGET_COLUMN = "class"
LABEL_COLUMN = "bioactivity_class"

# Feature-selection parameters
VARIANCE_THRESHOLD = 0.16
TOP_K_F_SCORE = 1000      # senior-style preselection before RFECV
FINAL_TOP_K_EXPORT = 200  # compact dataset for downstream modeling

# RFECV can be computationally expensive.
# Set RUN_RFECV = True when the low-variance/F-score output is ready.
RUN_RFECV = True

# RFECV speed/quality control
RFECV_STEP = 0.05         # fraction removed per iteration; faster than step=1
MIN_FEATURES_TO_SELECT = 20
MAX_FEATURES_FOR_RFECV = 1000

print("Input file:", INPUT_FILE)
print("Output directory:", OUTPUT_DIR)


## 2. Load RecA fingerprint modeling matrix

The input should contain metadata columns, a binary/integer target column named `class`, and PaDEL fingerprint descriptors such as PubChemFP, SubFP, Klekota-Roth, MACCS, or other binary descriptors.

In [ ]:
# ============================================================
# Load dataset
# ============================================================

if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {INPUT_FILE}\n"
        "Please run the previous PaDEL fingerprint/modeling-matrix notebook first."
    )

df = pd.read_csv(INPUT_FILE)
print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
# ============================================================
# Basic data validation
# ============================================================

if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found. "
        "Please make sure the modeling matrix contains a binary class column."
    )

available_meta_columns = [col for col in POSSIBLE_META_COLUMNS if col in df.columns]

# Remove duplicate metadata names while preserving order.
available_meta_columns = list(dict.fromkeys(available_meta_columns))

feature_columns = [col for col in df.columns if col not in available_meta_columns]

if TARGET_COLUMN in feature_columns:
    feature_columns.remove(TARGET_COLUMN)

meta = df[available_meta_columns].copy()

X = (
    df[feature_columns]
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df[TARGET_COLUMN].astype(int)

print("Detected metadata columns:", available_meta_columns)
print("Initial feature count:", X.shape[1])
print("Target distribution:")
print(y.value_counts().sort_index())

if LABEL_COLUMN in df.columns:
    print("\nBioactivity class distribution:")
    print(df[LABEL_COLUMN].value_counts())


## 3. Low-variance filtering

Descriptors with little or no variation are removed because they provide weak discriminative information and may introduce noise into machine-learning models.

In [ ]:
# ============================================================
# Low-variance filtering
# ============================================================

variance_selector = VarianceThreshold(threshold=VARIANCE_THRESHOLD)
X_low_array = variance_selector.fit_transform(X)

low_variance_features = X.columns[variance_selector.get_support()].tolist()

X_low = pd.DataFrame(
    X_low_array,
    columns=low_variance_features,
    index=X.index
)

low_variance_dataset = pd.concat([meta, X_low], axis=1)

LOW_VARIANCE_FILE = OUTPUT_DIR / "03_recA_low_variance_filtered.csv"
low_variance_dataset.to_csv(LOW_VARIANCE_FILE, index=False)

print("Variance threshold:", VARIANCE_THRESHOLD)
print("Feature count before filtering:", X.shape[1])
print("Feature count after filtering:", X_low.shape[1])
print("Saved:", LOW_VARIANCE_FILE)


## 4. ANOVA F-score feature ranking

The F-score is used to rank descriptors according to their ability to separate active and inactive RecA compounds. This step follows the senior workflow while keeping RecA-specific metadata and output names.

In [ ]:
# ============================================================
# ANOVA F-score ranking
# ============================================================

if X_low.shape[1] == 0:
    raise ValueError(
        "No descriptors remain after low-variance filtering. "
        "Try lowering VARIANCE_THRESHOLD."
    )

f_selector = SelectKBest(score_func=f_classif, k="all")
f_selector.fit(X_low, y)

feature_scores = pd.DataFrame({
    "feature": X_low.columns,
    "f_score": f_selector.scores_,
    "p_value": f_selector.pvalues_,
})

feature_scores = feature_scores.replace([np.inf, -np.inf], np.nan)
feature_scores["f_score"] = feature_scores["f_score"].fillna(-1.0)
feature_scores["p_value"] = feature_scores["p_value"].fillna(1.0)

feature_scores = feature_scores.sort_values(
    by=["f_score", "p_value"],
    ascending=[False, True]
).reset_index(drop=True)

FSCORE_FILE = OUTPUT_DIR / "03_recA_fscore_ranking.csv"
feature_scores.to_csv(FSCORE_FILE, index=False)

print("Saved:", FSCORE_FILE)
display(feature_scores.head(20))


In [ ]:
# ============================================================
# Export top F-score datasets
# ============================================================

top_k_for_rfecv = min(TOP_K_F_SCORE, len(feature_scores), MAX_FEATURES_FOR_RFECV)
top_k_final = min(FINAL_TOP_K_EXPORT, len(feature_scores))

top_features_for_rfecv = feature_scores.head(top_k_for_rfecv)["feature"].tolist()
top_features_final = feature_scores.head(top_k_final)["feature"].tolist()

X_fscore = X_low[top_features_for_rfecv].copy()

top_fscore_dataset = pd.concat([meta, X_low[top_features_final]], axis=1)
TOP_FSCORE_FILE = OUTPUT_DIR / "03_recA_top_fscore_dataset.csv"
top_fscore_dataset.to_csv(TOP_FSCORE_FILE, index=False)

print("Top features used for RFECV:", len(top_features_for_rfecv))
print("Top features exported for downstream modeling:", len(top_features_final))
print("Saved:", TOP_FSCORE_FILE)
display(top_fscore_dataset.head())


## 5. Publication-quality summary plots

These plots help document the feature-selection process in a manuscript, thesis, or supplementary material.

In [ ]:
# ============================================================
# F-score summary plot
# ============================================================

top_plot = feature_scores.head(30).copy()

plt.figure(figsize=(10, 8))
plt.barh(top_plot["feature"][::-1], top_plot["f_score"][::-1])
plt.xlabel("ANOVA F-score")
plt.ylabel("Descriptor")
plt.title("Top 30 RecA PaDEL Fingerprint Descriptors by F-score")
plt.tight_layout()

fig_path = FIGURE_DIR / "03_recA_top30_fscore_features.png"
plt.savefig(fig_path, dpi=300, bbox_inches="tight")
plt.show()

print("Saved figure:", fig_path)


## 6. RFECV benchmark using SVM, Random Forest, and XGBoost

This section combines the senior notebook's SVM-RFE, RF-RFE, and XGBoost-RFE idea with a more publication-ready implementation using `RFECV`.

To reduce data leakage risk, RFECV is performed inside cross-validation on the preselected F-score matrix. The final selected features are exported separately for each estimator.

In [ ]:
# ============================================================
# Helper functions for adaptive cross-validation and RFECV
# ============================================================

def make_cv(y, max_splits=10, random_state=RANDOM_STATE):
    """Create stratified CV with a safe number of folds based on class size."""
    counts = pd.Series(y).value_counts()
    min_class_count = int(counts.min())
    n_splits = max(2, min(max_splits, min_class_count))
    return StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)


def extract_rfecv_scores(rfecv):
    """Return a DataFrame of RFECV scores compatible with multiple sklearn versions."""
    if hasattr(rfecv, "cv_results_"):
        results = pd.DataFrame(rfecv.cv_results_)
        keep_cols = [c for c in ["n_features", "mean_test_score", "std_test_score"] if c in results.columns]
        return results[keep_cols].copy()

    # Older sklearn fallback
    if hasattr(rfecv, "grid_scores_"):
        return pd.DataFrame({
            "iteration": np.arange(1, len(rfecv.grid_scores_) + 1),
            "mean_test_score": rfecv.grid_scores_,
        })

    return pd.DataFrame()


def run_rfecv_model(model_name, estimator, X_input, y_input, scale=False):
    """Run RFECV and export selected features and CV scores."""
    cv = make_cv(y_input, max_splits=10)

    if scale:
        estimator_for_rfecv = Pipeline([
            ("scaler", StandardScaler()),
            ("model", estimator),
        ])
        importance_getter = "named_steps.model.coef_"
    else:
        estimator_for_rfecv = estimator
        importance_getter = "auto"

    rfecv = RFECV(
        estimator=estimator_for_rfecv,
        step=RFECV_STEP,
        min_features_to_select=min(MIN_FEATURES_TO_SELECT, X_input.shape[1]),
        cv=cv,
        scoring="accuracy",
        n_jobs=-1,
        importance_getter=importance_getter,
    )

    rfecv.fit(X_input, y_input)

    selected_features = X_input.columns[rfecv.support_].tolist()
    ranking = pd.DataFrame({
        "feature": X_input.columns,
        "selected": rfecv.support_,
        "ranking": rfecv.ranking_,
    }).sort_values(["selected", "ranking"], ascending=[False, True])

    selected_file = RFE_OUTPUT_DIR / f"03_recA_rfecv_selected_features_{model_name}.csv"
    ranking_file = RFE_OUTPUT_DIR / f"03_recA_rfecv_ranking_{model_name}.csv"
    score_file = RFE_OUTPUT_DIR / f"03_recA_rfecv_scores_{model_name}.csv"
    dataset_file = RFE_OUTPUT_DIR / f"03_recA_rfecv_dataset_{model_name}.csv"

    pd.DataFrame({"feature": selected_features}).to_csv(selected_file, index=False)
    ranking.to_csv(ranking_file, index=False)

    scores = extract_rfecv_scores(rfecv)
    if not scores.empty:
        scores.to_csv(score_file, index=False)

    selected_dataset = pd.concat([meta, X_input[selected_features]], axis=1)
    selected_dataset.to_csv(dataset_file, index=False)

    summary = {
        "model": model_name,
        "input_features": int(X_input.shape[1]),
        "selected_features": int(len(selected_features)),
        "best_cv_accuracy": float(getattr(rfecv, "best_score_", np.nan)) if hasattr(rfecv, "best_score_") else np.nan,
        "selected_file": str(selected_file),
        "ranking_file": str(ranking_file),
        "dataset_file": str(dataset_file),
    }

    return summary, rfecv, ranking


In [ ]:
# ============================================================
# Define RFECV estimators
# ============================================================

rfecv_estimators = {
    "svm_linear": {
        "estimator": LinearSVC(
            C=1.0,
            dual=False,
            max_iter=10000,
            random_state=RANDOM_STATE,
        ),
        "scale": True,
    },
    "random_forest": {
        "estimator": RandomForestClassifier(
            n_estimators=500,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            class_weight="balanced",
        ),
        "scale": False,
    },
}

if XGBOOST_AVAILABLE:
    rfecv_estimators["xgboost"] = {
        "estimator": XGBClassifier(
            n_estimators=500,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        "scale": False,
    }

print("RFECV models:", list(rfecv_estimators.keys()))
print("RFECV input matrix:", X_fscore.shape)


In [ ]:
# ============================================================
# Run RFECV benchmark
# ============================================================

rfecv_summaries = []
rfecv_objects = {}
rfecv_rankings = {}

if RUN_RFECV:
    for model_name, config in rfecv_estimators.items():
        print(f"\nRunning RFECV: {model_name}")
        summary, rfecv, ranking = run_rfecv_model(
            model_name=model_name,
            estimator=config["estimator"],
            X_input=X_fscore,
            y_input=y,
            scale=config["scale"],
        )
        rfecv_summaries.append(summary)
        rfecv_objects[model_name] = rfecv
        rfecv_rankings[model_name] = ranking
        print(summary)
else:
    print("RUN_RFECV is False. Skipping RFECV benchmark.")

summary_df = pd.DataFrame(rfecv_summaries)
SUMMARY_FILE = RFE_OUTPUT_DIR / "03_recA_rfecv_summary.csv"

if not summary_df.empty:
    summary_df.to_csv(SUMMARY_FILE, index=False)
    display(summary_df)
    print("Saved:", SUMMARY_FILE)


In [ ]:
# ============================================================
# Plot RFECV summary
# ============================================================

if RUN_RFECV and not summary_df.empty:
    plt.figure(figsize=(8, 5))
    plt.bar(summary_df["model"], summary_df["selected_features"])
    plt.xlabel("RFECV estimator")
    plt.ylabel("Number of selected descriptors")
    plt.title("Number of Selected RecA Descriptors by RFECV Estimator")
    plt.tight_layout()

    fig_path = FIGURE_DIR / "03_recA_rfecv_selected_feature_counts.png"
    plt.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()

    print("Saved figure:", fig_path)


## 7. Consensus feature set

For manuscript-level robustness, a consensus descriptor set can be generated by retaining features selected by at least two RFECV estimators. If only one RFECV estimator is available, its selected features are used.

In [ ]:
# ============================================================
# Build consensus RFECV feature set
# ============================================================

if RUN_RFECV and rfecv_rankings:
    selection_table = pd.DataFrame(index=X_fscore.columns)

    for model_name, ranking in rfecv_rankings.items():
        selected = ranking.set_index("feature")["selected"].astype(int)
        selection_table[model_name] = selected.reindex(X_fscore.columns).fillna(0).astype(int)

    selection_table["selection_count"] = selection_table.sum(axis=1)
    selection_table = selection_table.sort_values("selection_count", ascending=False)

    minimum_votes = 2 if len(rfecv_rankings) >= 2 else 1
    consensus_features = selection_table.query("selection_count >= @minimum_votes").index.tolist()

    if len(consensus_features) == 0:
        consensus_features = selection_table.head(FINAL_TOP_K_EXPORT).index.tolist()

    consensus_dataset = pd.concat([meta, X_fscore[consensus_features]], axis=1)

    CONSENSUS_TABLE_FILE = RFE_OUTPUT_DIR / "03_recA_rfecv_consensus_feature_votes.csv"
    CONSENSUS_DATASET_FILE = RFE_OUTPUT_DIR / "03_recA_rfecv_consensus_dataset.csv"

    selection_table.to_csv(CONSENSUS_TABLE_FILE)
    consensus_dataset.to_csv(CONSENSUS_DATASET_FILE, index=False)

    print("Number of consensus features:", len(consensus_features))
    print("Saved:", CONSENSUS_TABLE_FILE)
    print("Saved:", CONSENSUS_DATASET_FILE)
    display(selection_table.head(20))
else:
    print("Consensus feature set was not generated because RFECV was skipped.")


## 8. Final notes for publication

This notebook provides three feature-selection outputs that can be reported in a thesis or manuscript:

1. **Low-variance filtering** removes non-informative descriptors.
2. **ANOVA F-score ranking** identifies descriptors with strong class-separation power.
3. **RFECV with SVM, Random Forest, and XGBoost** evaluates model-dependent descriptor relevance.

For final QSAR/ML modeling, use one of these exported datasets:

- Conservative and fast option: `03_recA_top_fscore_dataset.csv`
- Model-driven option: `03_recA_rfecv_dataset_svm_linear.csv`, `03_recA_rfecv_dataset_random_forest.csv`, or `03_recA_rfecv_dataset_xgboost.csv`
- Robust consensus option: `03_recA_rfecv_consensus_dataset.csv`

In the manuscript, clearly state that feature selection was conducted only after the RecA bioactivity matrix was curated and converted into PaDEL molecular fingerprints.